# B2-022 — Session 5: Training and Using VAEs

*90 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260927`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:F6-svd-spectral`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C9-dimensionality-reduction`, `book1:C11-neural-training`, `B2-021-cross-modal-transformers-vision`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:F6-svd-spectral](../../../../book1/units/F6-svd-spectral/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C9-dimensionality-reduction](../../../../book1/units/C9-dimensionality-reduction/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

Session 4 produced the loss; this session turns it into a working model.
You will assemble a VAE module with an explicit `eps` argument, write a reproducible training loop that logs reconstruction and KL separately, avoid the reduction mistakes that silently change the objective, weight the KL with $\beta$, recognize posterior collapse from a loss trace, and use a trained VAE to sample new data from the prior and to interpolate in latent space.

## 1. The VAE module and its shape ledger

A VAE in this unit is one `nn.Module` with three methods:

* `encode(x) -> (mu, logvar)`: a shared hidden layer with ReLU, then two separate linear heads;
* `decode(z) -> x_hat`: an MLP from the latent width $k$ back to $d$ features, with a linear output;
* `forward(x, eps) -> (x_hat, mu, logvar)`: encode, reparameterize $z=\mu+\exp(\frac12\ell)\odot\varepsilon$, decode.

Shape ledger for $d=8$, hidden width $16$, $k=2$, batch $B$:

| tensor | shape | produced by |
|---|---|---|
| `x` | `(B,8)` | data rows |
| `h` | `(B,16)` | `relu(Linear(8,16)(x))` |
| `mu`, `logvar` | `(B,2)` each | `Linear(16,2)` heads |
| `eps` | `(B,2)` | drawn outside the module |
| `z` | `(B,2)` | `mu + torch.exp(0.5*logvar)*eps` |
| `x_hat` | `(B,8)` | `Linear(2,16)`, ReLU, `Linear(16,8)` |

**Why `eps` is an argument.** The module itself contains no randomness, so the same `(x, eps)` always gives the same output. The training loop owns the randomness and draws `eps` from a dedicated `torch.Generator` seeded once; evaluation passes a fixed `eps` tensor so before/after comparisons use identical noise. This makes training reproducible and makes bugs (such as a forgotten reparameterization) testable.

**Parameter registration.** Every layer must be an attribute (or inside an `nn.Sequential`) so that `model.parameters()` hands all of them to the optimizer. A head created inside `forward` would be re-initialized every call and never trained.

**Checkpoint 1A.** For $d=2$, hidden $16$, $k=2$, how many parameters do the two encoder heads have together?

**Checkpoint 1B.** Why must `forward` not call `torch.randn` itself in this unit's contract?

**Collected answers.** Each `Linear(16,2)` has $16\cdot2+2=34$ parameters, so $68$ together. Because the contract makes the module deterministic given `(x, eps)`: the loop's seeded generator controls the noise, evaluation can reuse a fixed `eps`, and tests can probe gradients with chosen noise.

## 2. The training loop and its loss trace

One full-batch update in this unit's order:

```python
eps = torch.randn(batch.shape[0], k, generator=eps_generator)   # fresh noise each update
optimizer.zero_grad(set_to_none=True)
x_hat, mu, logvar = model(batch, eps)
loss = negative_elbo(batch, x_hat, mu, logvar)                  # mean over batch of recon + KL
loss.backward()
optimizer.step()
```

**Log the parts, not only the total.** Record the batch means of `recon` and `KL` at every update, and check the bookkeeping identity $\text{total}=\overline{\text{recon}}+\overline{\text{KL}}$ to floating-point tolerance. A total that falls while KL is exactly $0$ tells a very different story from a total that falls while both parts move (Section 5).

**The trace is noisy.** Each update uses fresh $\varepsilon$, so the training loss jitters even at convergence. Compare averages over windows (for example, the mean of the last 20 updates against the first update), not two single noisy points.

**Held-out evaluation.** Under `model.eval()` (no effect here, but a good habit) and `torch.no_grad()`, compute the negative ELBO on held-out rows with a **fixed** `eps` tensor drawn once from its own seeded generator, before training and after training. Held-out rows never enter `backward()`. Evaluating with $\varepsilon=0$ (that is, $z=\mu$) is a different quantity, the reconstruction of the posterior mean; it is useful for plots but it is not the ELBO estimate.

**Checkpoint 2A.** Why compare a window mean of late training losses rather than the final update's loss?

**Checkpoint 2B.** Which identity should hold, update by update, between the logged numbers?

**Collected answers.** Fresh noise each update makes single losses jitter, so one late value can be unluckily high or low; a window average estimates the level reliably. The total equals the batch-mean reconstruction plus the batch-mean KL (to floating-point tolerance).

## 3. Loss-bookkeeping pitfalls: mean over features versus sum

The derivation fixes the reductions: **sum over features** in the reconstruction, **sum over latents** in the KL, **mean over the batch**. Each departure changes the objective, not just its scale.

* **Mean over features in the reconstruction.** Divides only the reconstruction by $d$. That is the same optimum as keeping the sum and multiplying the KL by $d$, a hidden $\beta=d$ (Section 4). With $d=8$ the KL is effectively eight times stronger, which often causes posterior collapse. Session 4's lab showed the batch loss changing from $0.639213$ to $0.482963$ on the same tensors.
* **Mean over latents in the KL.** Divides the KL by $k$: a hidden $\beta=1/k$, weakening the prior and making prior samples worse.
* **Sum over the batch instead of mean.** Multiplies the whole loss by $B$; with Adam this barely matters, but with plain SGD it multiplies the effective learning rate by $B$, and it makes losses incomparable across batch sizes.
* **`F.mse_loss` for the reconstruction.** Its default mean is over all $B\cdot d$ entries and it has no $\frac12$; `0.5 * d * F.mse_loss(...)` equals the correct batch-mean reconstruction.

**Reporting rule.** State the reduction next to every number: "negative ELBO, nats per example, constant $\frac d2\log2\pi$ dropped." A number without its reduction is not reproducible.

**Checkpoint 3A.** For $d=8$, write the batch-mean reconstruction term using `F.mse_loss`.

**Checkpoint 3B.** Which hidden $\beta$ does averaging the KL over $k=4$ latents introduce?

**Collected answers.** `0.5 * 8 * F.mse_loss(x_hat, x)` (equivalently `0.5 * ((x - x_hat)**2).sum(dim=1).mean()`). $\beta=1/4$.

## 4. $\beta$ weighting and KL warm-up

A **$\beta$-VAE** minimizes $\overline{\text{recon}}+\beta\,\overline{\mathrm{KL}}$ with a chosen $\beta>0$; the plain VAE is $\beta=1$.

* $\beta<1$ weakens the pull toward the prior. Reconstructions get sharper, but the aggregate of encoded codes may drift away from $N(0,I)$, so decoding **prior** samples can produce unrealistic outputs.
* $\beta>1$ strengthens the pull. Codes match the prior more closely and each latent coordinate tends to capture a separate factor, but reconstructions blur, and above some value the model stops using the latent at all (Section 5).
* For $\beta\ne1$ the objective is no longer a bound on $\log p(x)$; report it as a $\beta$-weighted objective, not as an ELBO.

**KL warm-up (annealing).** Start training with a small $\beta$ (often $0$) and raise it linearly to its final value over the first updates, for example $\beta_t=\min(1,t/T)$. Early on, the decoder learns to use the codes before the KL pressure arrives, which is a standard defence against collapse.

**Choosing $\beta$ honestly.** Treat $\beta$ as a hyperparameter: choose it with train rows only (or with a separate validation split carved from the train rows), and evaluate the chosen model once on held-out rows. Picking $\beta$ by held-out results is model-selection leakage.

**Checkpoint 4A.** Write $\beta_t$ for a warm-up that reaches $1$ after $T=100$ updates, and give $\beta_{25}$ and $\beta_{250}$.

**Checkpoint 4B.** Why should a $\beta=4$ objective not be reported as an ELBO?

**Collected answers.** $\beta_t=\min(1,t/100)$; $\beta_{25}=0.25$ and $\beta_{250}=1$. Because only $\beta=1$ gives the Jensen lower bound on $\log p(x)$; with $\beta=4$ the KL is over-counted and the number is a different objective.

## 5. Posterior collapse: symptoms and diagnosis from a trace

**Posterior collapse** is the failure in which the encoder's $q(z\mid x)$ becomes (almost) equal to the prior $N(0,I)$ for **every** input: the KL term drops to about $0$, the codes carry no information about $x$, and the decoder learns to output (roughly) the data mean regardless of $z$.

**Symptoms you can read from logs.**

1. Batch-mean KL near $0$ (for example below $0.01$ nats) and staying there.
2. Per-latent KL: computing the KL coordinate by coordinate shows which latents are **active** (clearly positive KL) and which are collapsed. Partial collapse, where some coordinates are active and others are not, is common and often harmless when the data needs fewer dimensions than $k$.
3. Encoded means `mu` nearly identical across very different inputs; `logvar` near $0$.
4. Reconstruction error stuck near the "predict the mean" baseline (the per-example variance sum).
5. Interpolations and prior samples that barely change as $z$ changes.

**Typical causes.** A KL weight that is too strong (large $\beta$, or a mean-over-features reduction acting as a hidden $\beta=d$), a decoder powerful enough to model the data without codes, or KL pressure arriving before the decoder has learned to use the codes.

**Remedies, in order of first resort.** Fix any reduction bug; lower $\beta$ (or use warm-up); then consider architecture changes. Always re-diagnose with the same logged quantities after the change.

**Checkpoint 5A.** A trace shows total loss falling from $3.1$ to $2.4$ while the batch-mean KL goes $0.40\to0.003$ and reconstruction stays near the mean baseline. Diagnosis?

**Checkpoint 5B.** Why is a collapsed second latent coordinate not necessarily a bug?

**Collected answers.** Posterior collapse: the drop comes from eliminating KL, not from better reconstruction; check reductions and reduce $\beta$ or add warm-up. If the data varies mainly along fewer directions than $k$, the model only needs that many active latents, and the rest can match the prior at no reconstruction cost.

## 6. Sampling from the prior and latent interpolation

**Generating new data.** A trained VAE generates by following its own generative story: draw $z\sim N(0,I_k)$ from a seeded generator, then decode, `x_new = model.decode(z)` under `torch.no_grad()`.
This unit reports the decoder **mean** $\hat x$ as the sample (it does not add the decoder's unit-variance noise), so samples are smoother than data: expect their per-feature spread to be somewhat smaller than the data's.
Useful statistical probes for a batch of decoded prior samples are: shape and dtype; all values finite; per-feature sample mean close to the training mean; per-feature sample standard deviation not collapsed (for example at least half the training standard deviation); and, for clustered data, each cluster receiving a reasonable share of samples.
Because these are statistics of random draws, every probe states a tolerance and the seed.

**Latent interpolation.** Encode two rows $x_a,x_b$ and use their posterior **means** $z_a=\mu(x_a)$, $z_b=\mu(x_b)$ (not random draws, which would add noise). For $t\in[0,1]$ take the straight line $z_t=(1-t)z_a+tz_b$, built with `t = torch.linspace(0, 1, m).unsqueeze(1)` so that `z_path` has shape `(m,k)` by broadcasting, and decode the whole path in one call. Exact probes: the path's first and last rows equal $z_a$ and $z_b$; consecutive differences are equal (a straight line at constant speed); decoded endpoints equal `decode(z_a)` and `decode(z_b)`.
A healthy VAE decodes a path that moves smoothly between the two inputs; a collapsed one decodes nearly the same point all along the path.

Encoding held-out rows for interpolation is evaluation, not training, as long as no gradient step or model-selection choice uses the result.

**Checkpoint 6A.** What shape does `z_path` have for $m=9$ steps and $k=2$, and what are its rows 0 and 8?

**Checkpoint 6B.** Why interpolate between posterior means rather than between random draws from $q$?

**Collected answers.** `(9,2)`; row 0 is $z_a$ and row 8 is $z_b$. Random draws add encoder noise to the endpoints, so the path would no longer be determined by the two inputs and could not pass exact endpoint probes.

## 7. Worked laboratory: train, diagnose, sample, interpolate

This lesson-only laboratory uses 50 two-cluster points in $\mathbb R^2$ generated in the cell from seed `20260927` (centers $(\pm1.5,0)$, spread $0.3$), split into 40 train and 10 held-out rows. It is not the unit generator's data, and its model, step count, and thresholds differ from p19 and p20.

**Protocol.** Seed `20260927` before building a `TinyVAE` with $d=2$, hidden $16$, $k=2$; Adam `lr=0.01`; 400 full-batch updates; `eps` from a generator seeded `20260927`; held-out evaluation with a fixed `eps` from a generator seeded `20260928`. The loss is $\overline{\text{recon}}+\beta\,\overline{\mathrm{KL}}$. Columns: held-out mean recon, held-out mean KL, and per-latent KL averaged over **training** rows.

| $\beta$ | update | held-out recon | held-out KL | per-latent KL (train) |
|---:|---:|---:|---:|---|
| 1 | 0 | 1.393 | 0.146 | 0.081, 0.052 |
| 1 | 50 | 1.294 | 0.181 | 0.146, 0.024 |
| 1 | 200 | 0.883 | 0.510 | 0.499, 0.002 |
| 1 | 400 | 0.986 | 0.541 | 0.532, 0.000 |
| 20 | 50 | 1.534 | 0.002 | 0.000, 0.000 |
| 20 | 200 | 1.434 | 0.000 | 0.000, 0.000 |
| 20 | 400 | 1.498 | 0.000 | 0.000, 0.000 |

**Diagnosis.** With $\beta=1$, one latent becomes active (KL about $0.53$) and the second collapses; that partial collapse is healthy here because the data varies mainly along one direction (training standard deviations $1.60$ and $0.31$). Held-out recon falls from $1.393$ to about $0.9$–$1.0$ and is noisy between checkpoints. With $\beta=20$, both latents collapse within 50 updates and reconstruction stays at the "predict the mean" level: textbook posterior collapse.

**Using the models.** Decoding 200 prior samples from the $\beta=1$ model gives a per-feature mean near $(0.03,0.03)$ (training mean $(-0.08,0.02)$), a first-feature spread of $1.36$, and about half the samples on each side of $x_1=0$, so both clusters are produced. Interpolating between the posterior means of the first two held-out rows decodes a path whose first feature runs $-1.42,-0.73,0.09,0.93,1.48$: a smooth trip between the clusters. The $\beta=20$ model decodes almost the same point, about $(-0.04,0.03)$, everywhere along the path, and its samples have spread about $0.10$.

**Checkpoint 7A.** From the table, what is the held-out KL after 400 updates at $\beta=1$, and which latent carries it?

**Checkpoint 7B.** Name two observations in the $\beta=20$ run that each independently indicate collapse.

**Collected answers.** About $0.541$ nats, carried almost entirely by the first latent (per-latent KL $0.532$ versus $0.000$). Any two of: KL near $0$ from update 50 on; reconstruction stuck near the mean baseline (about $1.5$, versus about $1$ at $\beta=1$); interpolation decoding a nearly constant point; prior samples with spread about $0.10$ instead of about $1.4$.

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F

torch.set_num_threads(1)
SEED = 20260927
data_gen = torch.Generator().manual_seed(SEED)
centers = torch.tensor([[-1.5, 0.0], [1.5, 0.0]])
points = centers[torch.arange(50) % 2] + 0.3 * torch.randn(50, 2, generator=data_gen)
train_x, heldout_x = points[:40], points[40:]

class TinyVAE(nn.Module):
    def __init__(self, in_dim=2, hidden=16, latent=2):
        super().__init__()
        self.hidden = nn.Linear(in_dim, hidden)
        self.mu_head = nn.Linear(hidden, latent)
        self.logvar_head = nn.Linear(hidden, latent)
        self.decoder = nn.Sequential(nn.Linear(latent, hidden), nn.ReLU(), nn.Linear(hidden, in_dim))
    def encode(self, x):
        h = F.relu(self.hidden(x))
        return self.mu_head(h), self.logvar_head(h)
    def decode(self, z):
        return self.decoder(z)
    def forward(self, x, eps):
        mu, logvar = self.encode(x)
        return self.decode(mu + torch.exp(0.5 * logvar) * eps), mu, logvar

def terms(x, x_hat, mu, logvar):
    recon = 0.5 * ((x - x_hat) ** 2).sum(dim=1)
    kl = 0.5 * (mu ** 2 + logvar.exp() - logvar - 1.0).sum(dim=1)
    return recon, kl

def train(beta, steps=400):
    torch.manual_seed(SEED)
    model = TinyVAE()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    eps_gen = torch.Generator().manual_seed(SEED)
    eval_eps = torch.randn(heldout_x.shape[0], 2, generator=torch.Generator().manual_seed(SEED + 1))
    rows = []
    for step in range(steps + 1):
        if step in (0, 50, 200, 400):
            with torch.no_grad():
                recon, kl = terms(heldout_x, *model(heldout_x, eval_eps))
                mu, logvar = model.encode(train_x)
                per_latent = (0.5 * (mu ** 2 + logvar.exp() - logvar - 1.0)).mean(dim=0)
            rows.append((step, round(recon.mean().item(), 3), round(kl.mean().item(), 3),
                         [round(v, 3) for v in per_latent.tolist()]))
        if step == steps:
            break
        eps = torch.randn(train_x.shape[0], 2, generator=eps_gen)
        optimizer.zero_grad(set_to_none=True)
        recon, kl = terms(train_x, *model(train_x, eps))
        (recon + beta * kl).mean().backward()
        optimizer.step()
    return model, rows

model_beta1, rows_beta1 = train(1.0)
for row in rows_beta1:
    print("beta=1", row)

In [ ]:
model_beta20, rows_beta20 = train(20.0)
for row in rows_beta20:
    print("beta=20", row)

for name, model in [("beta=1", model_beta1), ("beta=20", model_beta20)]:
    with torch.no_grad():
        z = torch.randn(200, 2, generator=torch.Generator().manual_seed(SEED + 2))
        samples = model.decode(z)
        mu_a, _ = model.encode(heldout_x[:1])
        mu_b, _ = model.encode(heldout_x[1:2])
        t = torch.linspace(0, 1, 5).unsqueeze(1)
        z_path = (1 - t) * mu_a + t * mu_b
        path = model.decode(z_path)
    assert samples.shape == (200, 2) and torch.isfinite(samples).all()
    assert torch.allclose(z_path[0], mu_a[0], atol=1e-6, rtol=1e-6) and torch.allclose(z_path[-1], mu_b[0], atol=1e-6, rtol=1e-6)
    print(name, "sample mean", [round(v, 3) for v in samples.mean(0).tolist()],
          "sample std", [round(v, 3) for v in samples.std(0).tolist()],
          "left share", round((samples[:, 0] < 0).float().mean().item(), 3))
    print(name, "decoded path, feature 1:", [round(v, 2) for v in path[:, 0].tolist()])

## 8. Common pitfalls and Exam connections

**Common pitfalls (each with its fix).**

1. *Reduction drift.* Mean over features or latents changes the objective (hidden $\beta$). Fix: sum, sum, then batch mean; assert total $=$ recon $+$ KL each update.
2. *Reading one noisy point.* Fix: window averages; fixed evaluation `eps`.
3. *Mistaking collapse for convergence.* A falling total with KL $\to0$ is not success. Fix: log per-latent KL and compare reconstruction with the mean baseline.
4. *Sampling by encoding data.* "Samples" made by encoding and decoding training rows are reconstructions, not generations. Fix: decode $z\sim N(0,I)$.
5. *Interpolating random draws.* Fix: interpolate posterior means.
6. *Training on held-out rows or tuning $\beta$ on them.* Fix: train rows only enter `backward()`; select hyperparameters on train-derived data.
7. *Non-differentiable sampling in the loop.* `.sample()` or `.detach()` on $z$. Fix: reparameterize (Session 4).

**Exam connections.** p19 trains a tiny VAE and certifies negative-ELBO decrease, KL bookkeeping, and held-out evaluation; p20 samples from the prior and interpolates with shape and statistic probes; p22 diagnoses posterior collapse from a literal trace and chooses a $\beta$ remedy; p23 repairs a loss with a sign error, a variance-versus-log-variance mix-up, and a reduction mismatch; p24 audits a pipeline for train/held-out leakage and a non-differentiable sampling step.

**Checkpoint 8A.** A notebook reports 'generated samples' produced by `model(train_x, eps)`. What is wrong?

**Checkpoint 8B.** Which practice gives you a literal KL/reconstruction trace to diagnose?

**Collected answers.** Those are reconstructions of training rows, not generations; generation decodes prior draws $z\sim N(0,I)$ with `model.decode`. p22.

## 9. Forward-only deeper topic: from VAEs to diffusion

Forward-only deeper topic.
VAE samples are often blurry because a unit-variance Gaussian decoder averages over plausible outputs.
Hierarchical VAEs stack several latent layers; vector-quantized VAEs use discrete codes; and `B2-023-generative-models-diffusion` replaces the single encoder step with a long fixed chain of small Gaussian noising steps and learns to reverse it, which can be read as a very deep VAE whose encoder is not learned.
GANs in B2-023 drop the likelihood altogether and train a generator against a discriminator.
None of these are prerequisites, permitted shortcuts, or assessed APIs in B2-022.

This final section is forward-only and not assessed.

**Checkpoint 9A.** Is a hierarchical or vector-quantized VAE required in p19 or p20?

**Checkpoint 9B.** In the diffusion view above, which VAE component is fixed rather than learned?

**Collected answers.** No; both practices pin a single-latent-layer `TinyVAE`. The encoder: the noising chain is fixed, and only the reverse (decoder-like) direction is learned.